# 🚀 Fine-Tuning Vaani AI on Google Colab (Free GPU)
This notebook fine-tunes **Vaani AI** (`Qwen/Qwen2.5-0.5B-Instruct` or larger) on a large dataset using **LoRA / PEFT** and saves the adapter.

### Steps:
1. Ensure GPU is enabled (**Runtime** > **Change runtime type** > Select **T4 GPU**)
2. Run all cells below to train the model and download `vaani-lora.zip`.

In [ ]:
# 1. Install Dependencies
!pip install -q -U transformers datasets trl peft accelerate bitsandbytes

In [ ]:
# 2. Download and Preprocess the Dataset (Dolly 15k or Alpaca)
from datasets import load_dataset
import json

print("Downloading databricks-dolly-15k from Hugging Face Hub...")
raw_data = load_dataset("databricks/databricks-dolly-15k", split="train")

instruction_prompt = (
    "You are Vaani AI (@vaaniai), an intelligent, concise, and helpful social media assistant. "
    "Solve the user's question clearly, accurately, and politely within tweet limits."
)

processed = []
for item in raw_data:
    q = item["instruction"].strip()
    if item.get("context"):
        q += f"\n\nContext: {item['context'].strip()}"
    resp = item["response"].strip()
    if q and resp and len(resp) <= 2500:
        processed.append({"instruction": instruction_prompt, "input": q, "output": resp})

with open("train_large.jsonl", "w", encoding="utf-8") as f:
    for p in processed:
        f.write(json.dumps(p) + "\n")

print(f"Processed {len(processed)} high-quality training examples!")

In [ ]:
# 3. Configure LoRA Training
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, TaskType
from trl import SFTTrainer
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "vaani-lora"

print(f"Loading base model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

train_dataset = load_dataset("json", data_files="train_large.jsonl", split="train")

def formatting_prompts_func(example):
    texts = []
    for inst, inp, out in zip(example["instruction"], example["input"], example["output"]):
        text = f"<|im_start|>system\n{inst}<|im_end|>\n<|im_start|>user\n{inp}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        texts.append(text)
    return texts

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=512,
)

print("Starting LoRA training on GPU...")
trainer.train()

# Save LoRA adapter
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}!")

In [ ]:
# 4. Zip and Download the Trained Adapter
import shutil
from google.colab import files

shutil.make_archive("vaani-lora", "zip", OUTPUT_DIR)
files.download("vaani-lora.zip")
print("Download started! Unzip this into your 'checkpoints/vaani-lora/' folder and set HF_LORA_PATH=checkpoints/vaani-lora in .env")